### Finding optimal hyperparameters with RandomizedSearchCV, classification data

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import label_binarize


In [ ]:
# load the credit group data, replace this with your own .csv -data
# this version of data should be READY TO GO to training
# which means => only numeric values in the dataset
df = pd.read_csv("processed_classification_credit_group.csv")
df.head()

,Delay_from_due_date,Num_of_Delayed_Payment,Num_Credit_Inquiries,Credit_Utilization_Ratio,Credit_History_Age,Amount_invested_monthly,Monthly_Balance,Credit_Mix,Age,Annual_Income,...,Outstanding_Debt,Total_EMI_per_month,Payment_of_Min_Amount_No,Payment_of_Min_Amount_Yes,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Credit_Score
0,3.0,7.0,4.0,26.822620,265.0,80.415295,312.494089,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
1,3.0,7.0,4.0,31.944960,265.0,118.280222,284.629162,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
2,3.0,7.0,4.0,28.609352,267.0,81.699521,331.209863,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
3,5.0,4.0,4.0,31.377862,268.0,199.458074,223.451310,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
4,6.0,4.0,4.0,24.797347,269.0,41.420153,341.489231,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2


In [ ]:
target = "Credit_Score"

# CatBoost requires we save our continuous and categorical variables separately into lists
categorical_variables = ["Credit_Mix", 'Payment_of_Min_Amount_No', 'Payment_of_Min_Amount_Yes',
                        'Payment_Behaviour_High_spent_Large_value_payments',
                        'Payment_Behaviour_High_spent_Medium_value_payments',
                        'Payment_Behaviour_High_spent_Small_value_payments',
                        'Payment_Behaviour_Low_spent_Large_value_payments',
                        'Payment_Behaviour_Low_spent_Medium_value_payments']

# the usual X/y -split
X = df.drop(target, axis=1)
y = df[target]

# usual train/test -split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# NOTE! SCALING => some of the algorithms require this
scaler = StandardScaler()

# create separate versions for the scaled data
# because we need both unscaled and scaled versions later
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Let's try finding better hyperparameter values for CatBoost this time

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

# Define parameter grid for RandomizedSearchCV
# use numpy etc. to define various options for the randomize search to try on
param_dist = {
    'iterations': np.linspace(300, 750, 25), 
    'learning_rate': np.linspace(0.03, 0.25, 15),  
    'depth': [4, 5, 6, 7], 
}

cat_model = cb.CatBoostClassifier(verbose=0)

# Set up RandomizedSearchCV for 75 iterations
random_search = RandomizedSearchCV(
    estimator=cat_model,
    param_distributions=param_dist,
    n_iter=60, 
    scoring='accuracy',
    cv=3,
    n_jobs=-1
)

# Fit the random search
random_search.fit(X_train, y_train, cat_features=categorical_variables)

# Print the best parameters
print("\nBest Parameters:", random_search.best_params_)

c:\Users\tuomas.valtanen\ADVML2026lectures\advancedml2026\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
129 fits failed out of a total of 180.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
6 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\tuomas.valtanen\ADVML2026lectures\advancedml2026\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\tuomas.valtanen\ADVML2026lectures\advancedml2026\.venv\Lib\site-packages\catboost\core.py", line 5245, in fit
    self._fit(X, y, cat_features, text_features, embedding_features, None, graph, sample_weight, None, N

Best Parameters: {'learning_rate': np.float64(0.14), 'iterations': np.float64(600.0), 'depth': 7}
